# Diabetic retinopathy grading — from feature extraction to a validated classifier

This notebook continues the feature-extraction work and turns it into a model that
can be defended in an interview.  It is a thin driver over the `dr` package; every
step below is library code with tests behind it (`pytest -q`).

**The label trap this notebook exists to avoid.**  In EyePACS, APTOS-2019,
Messidor-2 and IDRiD *every image lives in one flat directory* and the severity
grade lives in a CSV.  Taking the label from the parent folder therefore yields a
single class, and any accuracy printed afterwards is meaningless.  Cell 3 joins
images to the grading CSV by file stem and **refuses to continue** if fewer than two
classes survive.

**The three decisions that make the numbers trustworthy**

| decision | why |
| --- | --- |
| split by **patient**, not by image | both eyes of a patient share camera, illumination and disease state; a random split leaks near-duplicates and inflates every metric |
| score with **quadratic weighted kappa** | the grades are ordered — confusing 0 with 4 is not the same mistake as confusing 0 with 1 |
| always print the **baselines** | with 40 % of eyes healthy, "always predict 0" already looks respectable on accuracy |

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from dr.config import GRADE_NAMES, Config
from dr.utils import setup_logging

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid", context="notebook")
setup_logging("WARNING")

cfg = Config.load()          # configs/default.yaml
cfg.features.cache = True    # reuse anything already extracted
print(f"dataset={cfg.data.dataset}  images={cfg.raw_dir}  work size={cfg.features.work_size}px")

## 1. Cohort

`dr synth` renders a demo cohort so the notebook runs without redistributing patient
data.  To use a real dataset instead, point the config at it — the rest of the
notebook is unchanged:

```python
cfg.data.dataset = "aptos"                    # or eyepacs / messidor2 / idrid
cfg.paths.raw_dir = "/data/aptos/train_images"
cfg.paths.labels_csv = "/data/aptos/train.csv"
```

In [ ]:
from dr.data.datasets import build_manifest, summarise_manifest
from dr.data.synthetic import generate_dataset

if not any(cfg.raw_dir.rglob("*.jpg")):
    generate_dataset(cfg)

manifest, join_meta = build_manifest(cfg)
print("label source:", join_meta["labels_csv"])
print("joined:", join_meta["n_joined"], "| images without a label:",
      join_meta["n_images_without_label"], "| patient-level grouping:",
      join_meta["patient_level_grouping"])

summarise_manifest(manifest)

### The check the original notebook was missing

This is the guard rail: if the labels had come from the folder name, `nunique()`
below would be 1 and everything downstream would be noise.  The pipeline raises a
`LabelJoinError` in that case rather than training on a degenerate target.

In [ ]:
counts = manifest["grade"].value_counts().sort_index()
print(counts.to_string())
print(f"\n{len(manifest)} eyes | {manifest['group'].nunique()} patients | "
      f"{manifest['grade'].nunique()} classes")
assert manifest["grade"].nunique() >= 2, "Single class — take the labels from the CSV."

majority = counts.max() / counts.sum()
print(f"\nAlways predicting the majority grade already gives {majority:.1%} accuracy.")
print("That is the number every model below has to be compared against.")

## 2. What the descriptors actually measure

Each photograph is cropped to its field of view, resized, illumination-corrected
(Ben Graham's local-average subtraction) and CLAHE-equalised on the green channel —
the channel with the highest vessel and lesion contrast.  The panel below shows the
stages and the structures the lesion descriptors count.

In [ ]:
from dr.features.descriptors import structure_maps
from dr.features.preprocess import preprocess_image

worst = manifest.sort_values("grade", ascending=False).iloc[0]
img = preprocess_image(worst["image_path"], cfg.features, cfg.quality)
maps = structure_maps(img, cfg.features)

overlay = np.clip(img.rgb.copy(), 0, 1)
overlay[maps["vessels"]] = [0.15, 0.85, 1.0]
overlay[maps["dark"]] = [1.0, 0.2, 0.2]
overlay[maps["bright"]] = [1.0, 0.95, 0.2]

fig, axes = plt.subplots(1, 4, figsize=(16, 4.2))
for ax, panel, title in zip(
    axes,
    [np.clip(img.rgb, 0, 1), img.clahe, np.clip(img.illum, 0, 1), overlay],
    ["field of view", "CLAHE green", "illumination corrected",
     "vessels / dark / bright"],
):
    ax.imshow(panel, cmap="gray" if panel.ndim == 2 else None)
    ax.set_title(title, fontsize=11)
    ax.axis("off")
fig.suptitle(f"grade {int(worst['grade'])} — {GRADE_NAMES[int(worst['grade'])]}", y=1.02)
plt.show()

In [ ]:
from dr.features.extract import apply_quality_control, extract_dataset, select_feature_columns

features = extract_dataset(manifest, cfg)                 # parallel + cached
features, qc = apply_quality_control(
    features, cfg.data.drop_ungradable, cfg.quality.drop_quantile
)
columns = select_feature_columns(features)

print(f"{len(features)} usable images x {len(columns)} descriptors "
      f"({qc['dropped']} rejected by quality control)")
features[["image_id", "grade", "group"] + columns[:4]].head()

## 3. Splitting — the step that decides whether the numbers mean anything

Both eyes of a patient are in the table.  A random split puts one eye in train and
the other in test; the model can then recognise the *patient* (same camera, same
illumination, same pigmentation) instead of the pathology.  Everything below splits
on the patient id and stratifies on the grade at the same time.

In [ ]:
from dr.modeling.splits import group_stratified_holdout, verify_no_leakage

X = features[columns].to_numpy(dtype=float)
y = features["grade"].to_numpy(dtype=int)
groups = features["group"].astype(str).to_numpy()
labels = sorted(np.unique(y).tolist())

train_idx, test_idx = group_stratified_holdout(
    y, groups, test_size=cfg.modeling.test_size, seed=cfg.seed
)
verify_no_leakage(groups[train_idx], groups[test_idx])   # raises on any overlap

print(f"train {len(train_idx)} eyes / {pd.unique(groups[train_idx]).size} patients")
print(f"test  {len(test_idx)} eyes / {pd.unique(groups[test_idx]).size} patients")
print("\nclass balance is preserved on both sides:")
print(pd.DataFrame({
    "train": pd.Series(y[train_idx]).value_counts(normalize=True).sort_index().round(3),
    "test": pd.Series(y[test_idx]).value_counts(normalize=True).sort_index().round(3),
}))

## 4. Training

The original cell trained a logistic regression and a random forest and scored them
with balanced accuracy.  Four things change here:

1. **`StratifiedGroupKFold`** instead of `StratifiedKFold` — no patient spans a fold.
2. **Quadratic weighted kappa** as the tuning objective, because the target is ordinal.
3. **An ordinal model** (`ordinal_gbm`): regress the grade, then learn the cut points
   that maximise QWK on out-of-fold scores.
4. **Two baselines**, printed in the same table, so the reader can see the floor.

In [ ]:
from sklearn.model_selection import cross_val_score

from dr.modeling.metrics import (classification_metrics, ordinal_scores,
                                 quadratic_weighted_kappa, referable_metrics)
from dr.modeling.models import MODEL_DESCRIPTIONS, build_estimator
from dr.modeling.splits import make_cv, safe_n_splits

def qwk_scorer(labels):
    from sklearn.metrics import make_scorer
    return make_scorer(lambda a, b: quadratic_weighted_kappa(a, b, labels=labels))

cv = make_cv(safe_n_splits(y[train_idx], groups[train_idx], cfg.modeling.cv_folds),
             cfg.seed, grouped=True)

rows, fitted = [], {}
for name in ["dummy_frequent", "logreg", "random_forest", "hist_gbm", "ordinal_gbm"]:
    estimator, _ = build_estimator(name, seed=cfg.seed)
    cv_scores = cross_val_score(estimator, X[train_idx], y[train_idx],
                                groups=groups[train_idx], cv=cv,
                                scoring=qwk_scorer(labels), n_jobs=-1)
    estimator.fit(X[train_idx], y[train_idx])
    fitted[name] = estimator

    y_pred = estimator.predict(X[test_idx])
    metrics = classification_metrics(y[test_idx], y_pred, labels=labels)
    metrics |= referable_metrics(y[test_idx], ordinal_scores(estimator, X[test_idx]),
                                 cfg.modeling.referable_threshold_grade,
                                 cfg.modeling.target_sensitivity)
    rows.append({"model": name,
                 "cv_qwk": cv_scores.mean(), "cv_std": cv_scores.std(),
                 "test_qwk": metrics["qwk"],
                 "balanced_acc": metrics["balanced_accuracy"],
                 "f1_macro": metrics["f1_macro"],
                 "accuracy": metrics["accuracy"],
                 "referable_auroc": metrics["referable_auroc"]})

summary = pd.DataFrame(rows).sort_values("test_qwk", ascending=False)
summary.round(3)

### Is the winner really better than the baseline?

A point estimate on ~80 test eyes is not evidence.  The paired bootstrap resamples
the *same* eyes for both models, which removes the between-patient variance that
makes two independent intervals look inconclusive.

In [ ]:
from dr.modeling.metrics import bootstrap_ci, paired_bootstrap_test

best_name = summary.iloc[0]["model"]
best = fitted[best_name]
best_pred = best.predict(X[test_idx])
baseline_pred = fitted["dummy_frequent"].predict(X[test_idx])

metric = lambda a, b: quadratic_weighted_kappa(a, b, labels=labels)
point, low, high = bootstrap_ci(metric, y[test_idx], best_pred, n_resamples=2000, seed=cfg.seed)
test = paired_bootstrap_test(metric, y[test_idx], best_pred, baseline_pred,
                             n_resamples=2000, seed=cfg.seed)

print(f"{best_name}: QWK {point:.3f}  95% CI [{low:.3f}, {high:.3f}]")
print(f"vs majority baseline: dQWK {test['difference']:+.3f} "
      f"[{test['ci_low']:+.3f}, {test['ci_high']:+.3f}], p = {test['p_value']:.4f}")

### How much would a naive random split have flattered us?

This is the experiment that turns "I split by patient" from a claim into a
measurement.

In [ ]:
estimator, _ = build_estimator(best_name, seed=cfg.seed)
n_splits = safe_n_splits(y, groups, cfg.modeling.cv_folds)

grouped_cv = cross_val_score(estimator, X, y, groups=groups,
                             cv=make_cv(n_splits, cfg.seed, grouped=True),
                             scoring=qwk_scorer(labels), n_jobs=-1)
random_cv = cross_val_score(estimator, X, y,
                            cv=make_cv(n_splits, cfg.seed, grouped=False),
                            scoring=qwk_scorer(labels), n_jobs=-1)

print(f"patient-grouped CV QWK : {grouped_cv.mean():.3f} +- {grouped_cv.std():.3f}")
print(f"random CV QWK          : {random_cv.mean():.3f} +- {random_cv.std():.3f}")
print(f"optimism of the naive split: {random_cv.mean() - grouped_cv.mean():+.3f} QWK")

## 5. Where the errors are

The confusion matrix is read row-wise: the diagonal is per-class recall.  For a
screening task the cell that matters most is *true grade ≥ 2 predicted as 0* — a
patient sent home who needed an ophthalmologist.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

cm = confusion_matrix(y[test_idx], best_pred, labels=labels)
fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))
ticks = [f"{g} {GRADE_NAMES[g]}" for g in labels]
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
            xticklabels=ticks, yticklabels=ticks, ax=axes[0])
axes[0].set_title(f"{best_name} — counts")
sns.heatmap(cm / np.clip(cm.sum(axis=1, keepdims=True), 1, None), annot=True, fmt=".2f",
            cmap="Blues", vmin=0, vmax=1, cbar=False,
            xticklabels=ticks, yticklabels=ticks, ax=axes[1])
axes[1].set_title("row-normalised (recall on the diagonal)")
for ax in axes:
    ax.set_xlabel("predicted"); ax.set_ylabel("true")
    ax.tick_params(axis="x", rotation=30); ax.tick_params(axis="y", rotation=0)
plt.tight_layout(); plt.show()

print(classification_report(y[test_idx], best_pred, zero_division=0))

## 6. Which descriptors carry the signal

`feature_importances_` is tempting but misleading: it is computed on the training
set and is biased towards features with many possible split points.  Permutation
importance answers the question that matters — *how much held-out QWK is lost when
this descriptor is destroyed?*

In [ ]:
from sklearn.inspection import permutation_importance

from dr.features.extract import describe_feature, feature_group

result = permutation_importance(best, X[test_idx], y[test_idx],
                                scoring=qwk_scorer(labels), n_repeats=20,
                                random_state=cfg.seed, n_jobs=-1)
importance = (pd.DataFrame({"feature": columns,
                            "family": [feature_group(c) for c in columns],
                            "importance": result.importances_mean,
                            "std": result.importances_std})
              .sort_values("importance", ascending=False))

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5), width_ratios=[2, 1])
top = importance.head(15).iloc[::-1]
axes[0].barh(top["feature"], top["importance"], xerr=top["std"], color="#31698a")
axes[0].set_xlabel("drop in hold-out QWK when permuted")
axes[0].set_title(f"top descriptors — {best_name}")

family = importance[importance["importance"] > 0].groupby("family")["importance"].sum()
family = family.sort_values(ascending=False)
sns.barplot(x=family.to_numpy(), y=family.index, hue=family.index,
            palette="viridis", legend=False, ax=axes[1])
axes[1].set_title("by descriptor family")
plt.tight_layout(); plt.show()

for _, row in importance.head(6).iterrows():
    print(f"{row['feature']:<42} {row['importance']:+.4f}  {describe_feature(row['feature'])}")

## 7. The decision that actually gets deployed

A screening programme does not deploy a five-way grade; it deploys *refer or not*
(grade ≥ 2).  The threshold is not 0.5 — it is the lowest score that still reaches
the mandated sensitivity, and the specificity there determines the clinic's workload.

In [ ]:
from sklearn.metrics import precision_recall_curve, roc_curve

scores = ordinal_scores(best, X[test_idx])
positive = (y[test_idx] >= cfg.modeling.referable_threshold_grade).astype(int)
ref = referable_metrics(y[test_idx], scores, cfg.modeling.referable_threshold_grade,
                        cfg.modeling.target_sensitivity)

fpr, tpr, _ = roc_curve(positive, scores)
precision, recall, _ = precision_recall_curve(positive, scores)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
axes[0].plot(fpr, tpr, lw=2, label=f"AUROC = {ref['referable_auroc']:.3f}")
axes[0].plot([0, 1], [0, 1], "--", color="grey", lw=1)
axes[0].scatter([1 - ref["referable_specificity"]], [ref["referable_sensitivity"]],
                color="crimson", zorder=5,
                label=f"sens {ref['referable_sensitivity']:.2f} / spec {ref['referable_specificity']:.2f}")
axes[0].set_xlabel("1 - specificity"); axes[0].set_ylabel("sensitivity")
axes[0].set_title("referable DR — ROC"); axes[0].legend(loc="lower right")

axes[1].plot(recall, precision, lw=2, color="#7a5195",
             label=f"AP = {ref['referable_ap']:.3f}")
axes[1].axhline(positive.mean(), ls="--", color="grey", lw=1,
                label=f"prevalence = {positive.mean():.2f}")
axes[1].set_xlabel("recall"); axes[1].set_ylabel("precision")
axes[1].set_title("precision-recall"); axes[1].legend(loc="lower left")
plt.tight_layout(); plt.show()

print(f"at {ref['referable_sensitivity']:.0%} sensitivity: specificity "
      f"{ref['referable_specificity']:.3f}, PPV {ref['referable_ppv']:.3f}, "
      f"NPV {ref['referable_npv']:.3f}")

## 8. What this does and does not show

**Shown.** A handcrafted, fully interpretable representation carries a real ordinal
signal; the ordinal model beats both baselines by a margin whose confidence interval
excludes zero; the lesion descriptors — not the colour of the fundus — carry most of
that signal; and a naive random split would have overstated the result by a
measurable amount.

**Not shown.** Anything about real patients. The demo cohort is synthetic, and the
descriptors are proxies (a "microaneurysm count" is a count of small dark top-hat
responses, not a clinically validated lesion detection). Transportability to another
camera and population requires external validation. See `MODEL_CARD.md`.

The full run — nested cross-validation, feature-family ablation, bootstrap intervals
on every model, and all figures — is one command:

```bash
python -m dr all
```